In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

text = Path("data/tiny_shakespeare.txt").read_text(encoding="utf-8")
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)


In [ ]:
# Bigram model: predict next character from previous character only.
# This is the simplest language model (like nanoGPT lesson 1).

N = vocab_size
counts = torch.zeros((N, N), dtype=torch.float32)
for t in range(len(data) - 1):
    i, j = data[t].item(), data[t + 1].item()
    counts[i, j] += 1

bigram = counts + 1  # smoothing
bigram /= bigram.sum(dim=1, keepdim=True)
print("Bigram table shape:", bigram.shape)


In [ ]:
# Sample from the bigram model
g = torch.Generator().manual_seed(0)
idx = torch.tensor([[0]])
for _ in range(200):
    logits = bigram[idx[-1]]
    idx_next = torch.multinomial(logits, num_samples=1, generator=g)
    idx = torch.cat([idx, idx_next], dim=1)

print(decode(idx.squeeze().tolist()))
